# 6. Squat Rep Counting (Google Colab)

This notebook starts after step 5 has produced processed squat feature files.

Input:
- `squat_feature_index.csv`
- `squat_features/*.npy`

Output:
- predicted rep counts per video
- a prototype FSM-based squat counter
- basic evaluation against labeled squat counts

## Reason, Approach, Idea, Result Interpretation

**Reason for this section**
- This stage turns processed squat features into rep-count predictions.
- It is the first end-to-end counting baseline after pose and feature extraction are complete.

**Approach**
- Load the engineered squat feature files.
- Use an FSM-based counter driven by signals such as `knee_flex`.
- Compare predicted counts with labeled counts and export evaluation results.

**Core idea**
- Counting should be based on movement phases like `UP`, `DESCENDING`, `BOTTOM`, and `ASCENDING`.
- A simple FSM gives a strong baseline before moving to more complex temporal models.

**How to interpret results**
- `MAE` is the average rep-count error per video. Lower is better.
- `RMSE` highlights large-count failures more strongly than MAE.
- `Within-1 accuracy` shows how often the prediction is within one rep of the label.
- Train and valid should be interpreted separately when tuning thresholds and judging generalization.


## 0. Connect Google Drive

**Why this section exists**
This notebook reads the squat feature outputs from Drive and writes tuned counting results back to Drive, so the mount must exist before any path variables are defined.

**Approach / idea**
Mount Google Drive first, then define the Drive-based project root and downstream file paths used by the rep-counting pipeline.

**Result interpretation**
If the mount works here, the `1. Paths` section should point to accessible files under `/content/drive` rather than failing on missing paths.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 1. Paths

**Why this section exists**
Rep counting depends on the feature outputs from step 5, so the notebook needs explicit locations for the feature index, summaries, and result files.

**Approach / idea**
Set the Drive-based paths once at the top and use them consistently throughout the notebook.

**Result interpretation**
If these paths are wrong, the rest of the notebook will fail for file reasons rather than modeling reasons.


In [ ]:
from pathlib import Path

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')
ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data' / 'LLSP' / 'annotation_cleaned'

SQUAT_FEATURE_INDEX_CSV = ANNOTATION_DIR / 'squat_feature_index.csv'
RUN_SUMMARY_CSV = ANNOTATION_DIR / 'squat_feature_summary.csv'
REP_COUNT_RESULTS_CSV = ANNOTATION_DIR / 'squat_rep_count_results.csv'
REP_COUNT_METRICS_CSV = ANNOTATION_DIR / 'squat_rep_metrics_summary.csv'

print('ANNOTATION_DIR =', ANNOTATION_DIR)
print('SQUAT_FEATURE_INDEX_CSV =', SQUAT_FEATURE_INDEX_CSV)
print('RUN_SUMMARY_CSV =', RUN_SUMMARY_CSV)
print('REP_COUNT_RESULTS_CSV =', REP_COUNT_RESULTS_CSV)
print('REP_COUNT_METRICS_CSV =', REP_COUNT_METRICS_CSV)

## 2. Imports

**Why this section exists**
This notebook relies on numerical operations, result aggregation, and visualization for debugging the counter behavior.

**Approach / idea**
Load the required Python modules early so the later sections focus on counting logic and evaluation.

**Result interpretation**
If imports fail here, stop and fix the environment before interpreting any counting results.


In [ ]:
from dataclasses import dataclass
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 3. Load Feature Index

**Why this section exists**
The feature index is the list of squat examples that will be counted and evaluated in this stage.

**Approach / idea**
Load the index, confirm the files exist, and establish the working set before applying any FSM logic.

**Result interpretation**
A clean load here means the rep-counting results will reflect the model logic rather than missing feature artifacts.


In [ ]:
feature_index = pd.read_csv(SQUAT_FEATURE_INDEX_CSV)
feature_index.head()

In [ ]:
print('rows =', len(feature_index))
print('missing feature files =', int((~feature_index['feature_path'].map(lambda p: Path(p).exists())).sum()))

## 3b. Verify Input Artifacts

**Why this section exists**
If steps 5 and 6 point to different files, the rep metrics can stay unchanged even after upstream improvements.

**Approach / idea**
Inspect the exact Drive files that this notebook is reading, print modification times, and compare the loaded summary means with the expected step-5 outputs.

**Result interpretation**
If the printed means or paths do not match the latest step-5 run, stop here and fix the artifact paths before interpreting any rep-count metrics.


In [ ]:
run_summary = pd.read_csv(RUN_SUMMARY_CSV)
sample_feature_path = Path(feature_index.iloc[0]['feature_path'])
sample_arr = np.load(sample_feature_path)

print('SQUAT_FEATURE_INDEX_CSV =', SQUAT_FEATURE_INDEX_CSV)
print('RUN_SUMMARY_CSV =', RUN_SUMMARY_CSV)
print('sample_feature_path =', sample_feature_path)
print('sample_feature_exists =', sample_feature_path.exists())
print('sample_feature_shape =', sample_arr.shape)
print('index_mtime =', SQUAT_FEATURE_INDEX_CSV.stat().st_mtime)
print('summary_mtime =', RUN_SUMMARY_CSV.stat().st_mtime)
print('sample_feature_mtime =', sample_feature_path.stat().st_mtime)
print('\nsummary means from RUN_SUMMARY_CSV')
print(run_summary[['frames_valid', 'mean_conf']].mean(numeric_only=True))
print('\nfirst feature paths from loaded index')
print(feature_index['feature_path'].head().to_string(index=False))


## 4. Feature Schema

These columns match the output order from step 5.

**Why this section exists**
The counter assumes certain feature columns such as `knee_flex`, `hip_drop`, and `frame_valid` are present and correctly named.

**Approach / idea**
Declare the expected schema explicitly so feature loading and counter logic stay aligned.

**Result interpretation**
If the schema changes upstream, this section is where incompatibilities should surface clearly.


In [ ]:
FEATURE_COLUMNS = [
    'frame_idx',
    'left_knee_angle',
    'right_knee_angle',
    'avg_knee_angle',
    'knee_flex',
    'left_hip_angle',
    'right_hip_angle',
    'avg_hip_angle',
    'hip_center_y',
    'knee_center_y',
    'ankle_center_y',
    'hip_drop',
    'leg_extension',
    'hip_velocity',
    'frame_valid',
    'mean_conf',
]

len(FEATURE_COLUMNS)

## 5. Load One Feature File

**Why this section exists**
Before counting everything, verify that one processed feature file loads correctly and looks structurally sound.

**Approach / idea**
Open one sample file and inspect its shape and columns before running the FSM.

**Result interpretation**
If a single sample already looks malformed, full-batch evaluation would be misleading.


In [ ]:
def load_feature_frame(path: Path) -> pd.DataFrame:
    arr = np.load(path)
    if arr.ndim != 2 or arr.shape[1] != len(FEATURE_COLUMNS):
        raise ValueError(f'Expected [T, {len(FEATURE_COLUMNS)}], got {arr.shape} for {path}')
    return pd.DataFrame(arr, columns=FEATURE_COLUMNS)


sample_row = feature_index.iloc[0]
sample_df = load_feature_frame(Path(sample_row['feature_path']))
print('video =', sample_row['name'])
print('shape =', sample_df.shape)
sample_df.head()

## 6. Counter Parameters

**Why this section exists**
The FSM behavior is governed by threshold values, so the initial configuration needs to be explicit and easy to inspect.

**Approach / idea**
Define a baseline threshold dictionary that the notebook can use for inspection, batch counting, and later tuning.

**Result interpretation**
These values are a starting point, not the final answer; weak metrics later usually mean this section needs tuning.


In [ ]:
FSM_CFG = {
    'min_conf': 0.8,
    'min_valid_ratio': 0.5,
    'enter_down': 20.0,
    'enter_bottom': 55.0,
    'exit_bottom': 40.0,
    'back_to_up': 15.0,
    'min_bottom_frames': 2,
}

FSM_CFG

## 7. Squat FSM

**Why this section exists**
Rep counting here is implemented as a finite-state machine, so this section defines the actual counting logic.

**Approach / idea**
Use threshold-driven state transitions over the squat features to move through `UP`, `DESCENDING`, `BOTTOM`, and `ASCENDING`.

**Result interpretation**
If the predicted counts are wrong, this state logic and its thresholds are the primary mechanisms to inspect.


In [ ]:
@dataclass
class CountResult:
    pred_count: int
    state_trace: list[str]
    event_frames: list[int]


def count_squat_reps(features: pd.DataFrame, cfg: dict) -> CountResult:
    count = 0
    state = 'UP'
    state_trace = []
    event_frames = []
    bottom_frames = 0

    for i, row in features.iterrows():
        valid_frame = (row['frame_valid'] >= cfg['min_valid_ratio']) and (row['mean_conf'] >= cfg['min_conf'])
        knee_flex = float(row['knee_flex'])

        if not valid_frame:
            state_trace.append(state)
            continue

        if state == 'UP':
            bottom_frames = 0
            if knee_flex > cfg['enter_down']:
                state = 'DESCENDING'

        elif state == 'DESCENDING':
            if knee_flex > cfg['enter_bottom']:
                state = 'BOTTOM'
                bottom_frames = 1
            elif knee_flex < cfg['back_to_up']:
                state = 'UP'

        elif state == 'BOTTOM':
            if knee_flex > cfg['exit_bottom']:
                bottom_frames += 1
            else:
                if bottom_frames >= cfg['min_bottom_frames']:
                    state = 'ASCENDING'
                else:
                    state = 'DESCENDING'

        elif state == 'ASCENDING':
            if knee_flex < cfg['back_to_up']:
                count += 1
                event_frames.append(int(row['frame_idx']))
                state = 'UP'
                bottom_frames = 0
            elif knee_flex > cfg['enter_bottom']:
                state = 'BOTTOM'
                bottom_frames = 1

        state_trace.append(state)

    return CountResult(pred_count=count, state_trace=state_trace, event_frames=event_frames)

## 8. Inspect One Count Trace

**Why this section exists**
Single-example inspection helps you see exactly how the FSM behaves on real squat signals before trusting aggregate metrics.

**Approach / idea**
Run the counter on one sample, then plot the key signals, thresholds, and state transitions over time.

**Result interpretation**
Use this view to diagnose under-counting, over-counting, jitter sensitivity, or poor threshold placement.


In [ ]:
sample_result = count_squat_reps(sample_df, FSM_CFG)
true_count = float(sample_row['count'])

print('video =', sample_row['name'])
print('true_count =', true_count)
print('pred_count =', sample_result.pred_count)
print('event_frames =', sample_result.event_frames[:20])

In [ ]:
plot_df = sample_df.copy()
plot_df['state'] = sample_result.state_trace

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(plot_df['knee_flex'], label='knee_flex')
axes[0].axhline(FSM_CFG['enter_down'], linestyle='--', label='enter_down')
axes[0].axhline(FSM_CFG['enter_bottom'], linestyle='--', label='enter_bottom')
axes[0].axhline(FSM_CFG['back_to_up'], linestyle='--', label='back_to_up')
for frame in sample_result.event_frames:
    axes[0].axvline(frame, color='green', alpha=0.3)
axes[0].legend()
axes[0].set_ylabel('knee_flex')

axes[1].plot(plot_df['hip_drop'], label='hip_drop')
axes[1].plot(plot_df['hip_velocity'], label='hip_velocity')
axes[1].legend()
axes[1].set_ylabel('motion signal')

state_to_num = {'UP': 0, 'DESCENDING': 1, 'BOTTOM': 2, 'ASCENDING': 3}
axes[2].plot(plot_df['state'].map(state_to_num), label='state')
axes[2].plot(plot_df['frame_valid'], label='frame_valid')
axes[2].legend()
axes[2].set_ylabel('state / valid')
axes[2].set_xlabel('frame')

plt.tight_layout()
plt.show()

## 9. Batch Counting

**Why this section exists**
This section applies the current counter to the full squat subset and creates the result table used for evaluation.

**Approach / idea**
Loop over the feature index, run the FSM per video, and save per-video predictions and metadata.

**Result interpretation**
The batch output is the raw evidence for whether the current counter is usable or still needs tuning.


In [ ]:
results = []

for i, row in feature_index.iterrows():
    features = load_feature_frame(Path(row['feature_path']))
    count_result = count_squat_reps(features, FSM_CFG)
    true_count = float(row['count']) if pd.notna(row['count']) else np.nan
    pred_count = float(count_result.pred_count)
    abs_error = abs(pred_count - true_count) if pd.notna(true_count) else np.nan

    results.append({
        'name': row['name'],
        'split': row.get('split', ''),
        'feature_path': row['feature_path'],
        'true_count': true_count,
        'pred_count': pred_count,
        'abs_error': abs_error,
        'events_found': len(count_result.event_frames),
        'frames_total': len(features),
        'frames_valid': int(features['frame_valid'].sum()),
        'mean_conf': float(features['mean_conf'].mean()),
    })

    if (i + 1) % 25 == 0 or (i + 1) == len(feature_index):
        print(f'[{i + 1}/{len(feature_index)}] counted')

results_df = pd.DataFrame(results)
results_df.to_csv(REP_COUNT_RESULTS_CSV, index=False)
print('saved =', REP_COUNT_RESULTS_CSV)

## 10. Evaluation

**Why this section exists**
A rep counter is only useful if its errors are measured, so this section turns raw predictions into interpretable metrics.

**Approach / idea**
Compute aggregate error metrics and inspect worst examples to understand both average performance and failure modes.

**Result interpretation**
Treat these numbers as baseline quality indicators; they show whether the current FSM is good enough or still exploratory.


In [ ]:
results_df.head()

In [ ]:
print('MAE overall =', results_df['abs_error'].mean())
print('RMSE overall =', np.sqrt(np.mean((results_df['pred_count'] - results_df['true_count']) ** 2)))
print('Within-1 accuracy =', np.mean(results_df['abs_error'] <= 1.0))

In [ ]:
results_df.groupby('split')[['abs_error']].mean()

In [ ]:
worst = results_df.sort_values('abs_error', ascending=False).head(15)
worst[['name', 'split', 'true_count', 'pred_count', 'abs_error', 'mean_conf', 'frames_valid']]

## 11. Tune FSM on Train Split

This section tunes a small grid of FSM thresholds on the `train` split only. That keeps threshold selection separate from the `valid` split, which is used as the first real generalization check.

**Why this section exists**
Threshold tuning should happen on `train` so `valid` can remain the first honest check of generalization.

**Approach / idea**
Search a small grid of candidate threshold combinations, rank them by train performance, and keep the best configuration.

**Result interpretation**
Improvement here means the FSM can be calibrated; overfitting will show up if valid does not improve afterward.


In [ ]:
TUNING_RESULTS_CSV = ANNOTATION_DIR / 'squat_rep_tuning_results.csv'
TUNED_REP_COUNT_RESULTS_CSV = ANNOTATION_DIR / 'squat_rep_count_results_tuned.csv'

def summarize_results(df: pd.DataFrame) -> dict:
    return {
        'rows': int(len(df)),
        'mae': float(df['abs_error'].mean()),
        'rmse': float(np.sqrt(np.mean((df['pred_count'] - df['true_count']) ** 2))),
        'within_1': float(np.mean(df['abs_error'] <= 1.0)),
    }


def evaluate_counter(index_df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    rows = []

    for _, row in index_df.iterrows():
        features = load_feature_frame(Path(row['feature_path']))
        count_result = count_squat_reps(features, cfg)
        true_count = float(row['count']) if pd.notna(row['count']) else np.nan
        pred_count = float(count_result.pred_count)
        rows.append({
            'name': row['name'],
            'split': row.get('split', ''),
            'feature_path': row['feature_path'],
            'true_count': true_count,
            'pred_count': pred_count,
            'abs_error': abs(pred_count - true_count) if pd.notna(true_count) else np.nan,
            'events_found': len(count_result.event_frames),
            'frames_total': len(features),
            'frames_valid': int(features['frame_valid'].sum()),
            'mean_conf': float(features['mean_conf'].mean()),
        })

    return pd.DataFrame(rows)


train_index = feature_index[feature_index['split'] == 'train'].reset_index(drop=True)
valid_index = feature_index[feature_index['split'] == 'valid'].reset_index(drop=True)

param_grid = {
    'min_conf': [0.7, 0.8, 0.9],
    'min_valid_ratio': [0.35, 0.5, 0.65],
    'enter_down': [15.0, 20.0, 25.0],
    'enter_bottom': [45.0, 55.0, 65.0],
    'exit_bottom': [30.0, 40.0, 50.0],
    'back_to_up': [10.0, 15.0, 20.0],
    'min_bottom_frames': [2, 3, 4],
}

candidate_rows = []

for values in product(*param_grid.values()):
    cfg = dict(FSM_CFG)
    cfg.update(dict(zip(param_grid.keys(), values)))

    if not (cfg['back_to_up'] < cfg['enter_down'] < cfg['exit_bottom'] < cfg['enter_bottom']):
        continue

    train_results = evaluate_counter(train_index, cfg)
    train_summary = summarize_results(train_results)
    candidate_rows.append({
        **cfg,
        'train_mae': train_summary['mae'],
        'train_rmse': train_summary['rmse'],
        'train_within_1': train_summary['within_1'],
    })

tuning_df = pd.DataFrame(candidate_rows).sort_values(['train_mae', 'train_rmse', 'train_within_1'], ascending=[True, True, False]).reset_index(drop=True)
tuning_df.to_csv(TUNING_RESULTS_CSV, index=False)

print('candidates =', len(tuning_df))
print('saved =', TUNING_RESULTS_CSV)
tuning_df.head(10)


In [ ]:
BEST_FSM_CFG = dict(FSM_CFG)
BEST_FSM_CFG.update(tuning_df.iloc[0][list(param_grid.keys())].to_dict())
BEST_FSM_CFG['min_bottom_frames'] = int(BEST_FSM_CFG['min_bottom_frames'])

BEST_FSM_CFG


## 12. Evaluate Tuned Counter

This reruns the counter with the best train-selected thresholds, but the main reported metric is the `valid` split. Train remains diagnostic only and the combined set is secondary context.

**Why this section exists**
After selecting thresholds on train, the next question is whether those thresholds actually help on unseen validation videos.

**Approach / idea**
Rerun the batch evaluation with the tuned configuration, but surface `valid` first as the headline metric and keep `train` as supporting diagnostic context.

**Result interpretation**
The `valid` split is the main score to report. If train improves but valid does not, the tuning is not robust enough yet.


In [ ]:
tuned_results_df = evaluate_counter(feature_index, BEST_FSM_CFG)
tuned_results_df.to_csv(TUNED_REP_COUNT_RESULTS_CSV, index=False)
print('saved =', TUNED_REP_COUNT_RESULTS_CSV)
tuned_results_df.head()


In [ ]:
baseline_summary = summarize_results(results_df)
tuned_summary = summarize_results(tuned_results_df)

comparison_df = pd.DataFrame([
    {'run': 'baseline', **baseline_summary},
    {'run': 'tuned', **tuned_summary},
])
comparison_df


In [ ]:
split_metrics = []
for split_name, split_df in tuned_results_df.groupby('split'):
    summary = summarize_results(split_df)
    split_metrics.append({'split': split_name, **summary})

pd.DataFrame(split_metrics).sort_values('split').reset_index(drop=True)


### Valid-First Reporting

Use the `valid` split as the main rep-counting result. The train split is useful for threshold selection and debugging, but it should not be the headline performance number.

In [ ]:
split_metrics_df = pd.DataFrame(split_metrics).sort_values('split').reset_index(drop=True)
valid_metrics = split_metrics_df[split_metrics_df['split'] == 'valid'].iloc[0].to_dict()
train_metrics = split_metrics_df[split_metrics_df['split'] == 'train'].iloc[0].to_dict()

print('Primary reported metric: VALID')
print('valid rows =', int(valid_metrics['rows']))
print('valid MAE =', valid_metrics['mae'])
print('valid RMSE =', valid_metrics['rmse'])
print('valid Within-1 =', valid_metrics['within_1'])

print('\nDiagnostic only: TRAIN')
print('train rows =', int(train_metrics['rows']))
print('train MAE =', train_metrics['mae'])
print('train RMSE =', train_metrics['rmse'])
print('train Within-1 =', train_metrics['within_1'])


In [ ]:
report_df = pd.DataFrame([
    {'role': 'primary_report', **valid_metrics},
    {'role': 'diagnostic_train', **train_metrics},
])
report_df.to_csv(REP_COUNT_METRICS_CSV, index=False)
print('saved =', REP_COUNT_METRICS_CSV)
report_df


In [ ]:
tuned_worst_valid = tuned_results_df[tuned_results_df['split'] == 'valid'].sort_values('abs_error', ascending=False).head(10)
tuned_worst_valid[['name', 'true_count', 'pred_count', 'abs_error', 'mean_conf', 'frames_valid']]


## 13. Next Tuning Directions

If the tuned counter is still weak on `valid`, the next improvements should focus on signal quality and state constraints rather than expanding the grid blindly.

- Add dwell-time constraints for `DESCENDING` and `ASCENDING`.
- Gate transitions with `hip_velocity` so small knee-flex spikes do not create false reps.
- Require a minimum squat depth using both `knee_flex` and `hip_drop`.
- Inspect the worst `valid` examples before adding more thresholds.


**Why this section exists**
Even after one tuning pass, the remaining errors need a concrete next-step plan rather than ad hoc trial and error.

**Approach / idea**
List the highest-value changes to try next based on the observed failure modes and the current FSM design.

**Result interpretation**
Use this section as the transition from baseline experimentation to a more targeted second iteration of the counter.
